# Portfolio Project 2 — Статистична значущість A/B-тестів

**Автор:** Юлія Бутенко

**Мета проєкту:** проаналізувати результати A/B-тестування за допомогою статистичних методів у Python та створити візуалізацію ключових конверсійних метрик у Tableau.

**Посилання:**
- 📊 Дашборд у Tableau: *[вставити посилання]*
- 📁 CSV з результатами розрахунків: *[вставити посилання]*

## Етапи проєкту
1. Завантаження даних напряму з BigQuery (датасет «DA»)
2. Перевірка даних
3. Налаштування метрик
4. Розрахунок статистичної значущості — у тоталі по тесту та в розрізах (пристрій, континент, країна, канал)
5. Фінальна таблиця результатів (формат файлу з умови завдання)
6. Збереження CSV для Tableau
7. Висновки

## 1. Завантаження даних з BigQuery

Дані беруться напряму з BigQuery, тому блокнот не залежить від файлів на Google Drive.

Для кожної комбінації *дата × країна × пристрій × континент × канал × тест × група* рахуємо:
- `session` — кількість сесій (знаменник усіх метрик);
- `add_payment_info`, `add_shipping_info`, `begin_checkout` — **кількість подій**;
- `new accounts` — кількість нових акаунтів.

👉 Під час запуску Google попросить дозвіл на доступ до акаунта: натисніть **«Дозволити»**.

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

PROJECT_ID = 'data-analytics-mate'
client = bigquery.Client(project=PROJECT_ID)

QUERY = '''
WITH s AS (
  SELECT
    se.date, se.ga_session_id,
    sp.country, sp.device, sp.continent, sp.channel,
    ab.test, ab.test_group
  FROM `data-analytics-mate.DA.ab_test` ab
  JOIN `data-analytics-mate.DA.session` se USING (ga_session_id)
  JOIN `data-analytics-mate.DA.session_params` sp USING (ga_session_id)
),
sessions AS (
  SELECT date, country, device, continent, channel, test, test_group,
         'session' AS event_name, COUNT(DISTINCT ga_session_id) AS value
  FROM s
  GROUP BY 1, 2, 3, 4, 5, 6, 7
),
events AS (
  SELECT s.date, s.country, s.device, s.continent, s.channel, s.test, s.test_group,
         ep.event_name, COUNT(*) AS value
  FROM s
  JOIN `data-analytics-mate.DA.event_params` ep USING (ga_session_id)
  WHERE ep.event_name IN ('add_payment_info', 'add_shipping_info', 'begin_checkout')
  GROUP BY 1, 2, 3, 4, 5, 6, 7, 8
),
accounts AS (
  SELECT s.date, s.country, s.device, s.continent, s.channel, s.test, s.test_group,
         'new accounts' AS event_name, COUNT(DISTINCT acs.account_id) AS value
  FROM s
  JOIN `data-analytics-mate.DA.account_session` acs USING (ga_session_id)
  GROUP BY 1, 2, 3, 4, 5, 6, 7
)
SELECT * FROM sessions
UNION ALL SELECT * FROM events
UNION ALL SELECT * FROM accounts
'''

df = client.query(QUERY).to_dataframe()
print('Розмір датасету:', df.shape)
df.head()

## 2. Перевірка даних

Сума значень по кожному тесту, групі та події.

In [ ]:
check = (df.groupby(['test', 'test_group', 'event_name'])['value']
           .sum().unstack('event_name'))
check

## 3. Налаштування метрик

Статично задаються **лише** назви 4 метрик і те, як вони рахуються (подія / сесії). Усе інше — тести, групи, розрізи — визначається з даних автоматично в циклах. Щоб додати нову метрику, достатньо дописати один рядок у словник `METRICS`, а щоб додати розріз — одне слово в `SEGMENTS`.

In [ ]:
DENOM = 'session'                      # знаменник для всіх метрик
METRICS = {                            # назва метрики : подія-чисельник
    'add_payment_info / session':  'add_payment_info',
    'add_shipping_info / session': 'add_shipping_info',
    'begin_checkout / session':    'begin_checkout',
    'new_accounts / session':      'new accounts',
}
SEGMENTS = ['total', 'device', 'continent', 'country', 'channel']
CONTROL, TREATMENT = 1, 2              # 1 — контрольна група, 2 — тестова
ALPHA = 0.05                           # рівень значущості

df['total'] = 'Total'                  # штучний розріз для розрахунку в тоталі
EVENTS = list(METRICS.values()) + [DENOM]
print('Налаштування готові')

## 4. Розрахунок статистичної значущості

**Логіка:**
- Метрика = кількість подій / кількість сесій (наприклад, `add_payment_info / session`).
- Метрики контрольної (група 1) та тестової (група 2) груп порівнюються **z-тестом для двох пропорцій** (`proportions_ztest` з бібліотеки statsmodels).
- Нульова гіпотеза H₀: метрики в групах однакові.
- Якщо **p-value < 0.05**, відхиляємо H₀ → різниця **статистично значуща**:
  - метрика тестової групи вища → `Significant increase`
  - метрика тестової групи нижча → `Significant decrease`
- Якщо p-value ≥ 0.05 → `Not significant`.
- `metric_change` = (метрика тест − метрика контроль) / метрика контроль × 100 %.

Розрахунок іде трьома вкладеними циклами: **розріз → (тест, значення розрізу) → метрика**.

In [ ]:
results = []

for seg in SEGMENTS:
    # сума по кожній події для кожного тесту / значення розрізу / групи
    agg = (df[df['event_name'].isin(EVENTS)]
           .groupby(['test', seg, 'test_group', 'event_name'])['value'].sum()
           .unstack('event_name', fill_value=0))

    for (test, seg_value), g in agg.groupby(level=[0, 1]):
        g = g.droplevel([0, 1])
        if not {CONTROL, TREATMENT}.issubset(g.index):   # потрібні обидві групи
            continue
        ctrl, trt = g.loc[CONTROL], g.loc[TREATMENT]

        for metric, event in METRICS.items():
            n1, n2 = ctrl.get(DENOM, 0), trt.get(DENOM, 0)   # сесії
            x1, x2 = ctrl.get(event, 0), trt.get(event, 0)   # події
            if n1 == 0 or n2 == 0:
                continue

            cr1, cr2 = x1 / n1, x2 / n2
            z, p = proportions_ztest([x2, x1], [n2, n1])
            if np.isnan(p):          # обидві метрики = 0 → різниці немає
                z, p = 0.0, 1.0

            significant = p < ALPHA
            if not significant:
                label = 'Not significant'
            else:
                label = 'Significant increase' if cr2 > cr1 else 'Significant decrease'

            results.append({
                'test': test, 'segment_type': seg, 'segment_value': seg_value,
                'metric': metric, 'numerator_event': event,
                'sessions_control': n1, 'sessions_test': n2,
                'events_control': x1, 'events_test': x2,
                'cr_control': cr1, 'cr_test': cr2,
                'uplift_pct': (cr2 - cr1) / cr1 * 100 if cr1 > 0 else np.nan,
                'z_stat': z, 'p_value': p,
                'is_significant': significant, 'result': label,
            })

res = pd.DataFrame(results)
print('Усього рядків з результатами:', len(res))

## 5. Фінальна таблиця: результати в тоталі по кожному тесту

Формат таблиці відповідає файлу з умови завдання.

In [ ]:
total = res[res['segment_type'] == 'total']
final = pd.DataFrame({
    'test_number':               total['test'],
    'metric':                    total['metric'],
    'numerator_event':           total['numerator_event'],
    'denominator_event':         DENOM,
    'numerator_count_test':      total['events_test'],
    'denominator_count_test':    total['sessions_test'],
    'conversion_rate_test':      total['cr_test'],
    'numerator_count_control':   total['events_control'],
    'denominator_count_control': total['sessions_control'],
    'conversion_rate_control':   total['cr_control'],
    'metric_change':             total['uplift_pct'],
    'z_stat':                    total['z_stat'],
    'p_value':                   total['p_value'],
    'significant':               total['is_significant'],
}).sort_values(['test_number', 'metric']).reset_index(drop=True)
final

## 6. Значущі результати в розрізах

⚠️ У малих сегментах (наприклад, країни з кількома десятками сесій) результати ненадійні, тому враховуємо лише сегменти з ≥ 1000 сесій у кожній групі.

In [ ]:
MIN_SESSIONS = 1000
res['enough_data'] = (res['sessions_control'] >= MIN_SESSIONS) & (res['sessions_test'] >= MIN_SESSIONS)

summary = (res[res['enough_data']]
           .groupby(['segment_type', 'result']).size()
           .unstack(fill_value=0))
display(summary)

sig = res[res['is_significant'] & res['enough_data'] & (res['segment_type'] != 'total')]
sig[['test', 'segment_type', 'segment_value', 'metric',
     'cr_control', 'cr_test', 'uplift_pct', 'p_value', 'result']].sort_values('p_value').reset_index(drop=True)

## 7. Збереження результатів

- `ab_test_results_final.csv` — фінальна таблиця в тоталі (формат з умови завдання);
- `ab_test_results.csv` — усі розрізи, використовується в дашборді Tableau.

In [ ]:
from google.colab import files

final.to_csv('ab_test_results_final.csv', index=False)
res.to_csv('ab_test_results.csv', index=False)

files.download('ab_test_results_final.csv')
files.download('ab_test_results.csv')

## 8. Висновки

*[заповнити після запуску на основі таблиць з розділів 5 і 6]*